In [2]:
# One Hot Encoding
# Interaction Feature

### One Hot Encoding :- This is a method used in machine learning to convert categorical data into a numerical format. It creates binary columns for each category, where each column represents a unique category and contains a 1 or 0 indicating the presence or absence of that category in the data. This technique is particularly useful for algorithms that require numerical input, such as linear regression or neural networks.

In [2]:
import pandas as pd 
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [4]:
insurance_data = pd.read_csv("01_insurance.csv")
insurance_data.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [5]:
# Features (X) and target (y)
X = insurance_data.drop(columns=["charges"])
y = insurance_data["charges"]

# One-Hot Encode 'region' (avoid dummy variable trap)
X = pd.get_dummies(
    X,
    columns=["region"],
    drop_first=True,
    dtype=int
)

# Binary encoding for categorical features
X["sex"] = X["sex"].map({"female": 1, "male": 0})
X["smoker"] = X["smoker"].map({"yes": 1, "no": 0})

In [6]:
X.head()
## North_East = 1 - (Northwest + Southeast + southWest)
# X[(X["region_northwest"] == 0) &  
#   (X["region_southeast"] == 0) &
#   (X["region_southwest"] == 0)]

,age,sex,bmi,children,smoker,region_northwest,region_southeast,region_southwest
0,19,1,27.900,0,1,0,0,1
1,18,0,33.770,1,0,0,1,0
2,28,0,33.000,3,0,0,1,0
3,33,0,22.705,0,0,1,0,0
4,32,0,28.880,0,0,1,0,0


In [13]:
# Split data into training and testing sets
# 80% train, 20% test | random_state=42 for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [14]:
# Train Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [15]:
# Predict insurance charges on test data
y_pred = model.predict(X_test)

In [16]:
# Convert predictions to Series for easy viewing
pd.Series(y_pred).head()

0     8969.550274
1     7068.747443
2    36858.410912
3     9454.678501
4    26973.173457
dtype: float64

In [17]:
# Evaluate model performance using R² score
r2 = r2_score(y_test, y_pred)
print("R² Score:", r2)

R² Score: 0.7835929767120724


In [18]:
# Adjusted R²: penalizes unnecessary features
n_samples = X_test.shape[0]
n_features = X_test.shape[1]

adjusted_r2 = 1 - ((1 - r2) * (n_samples - 1) / (n_samples - n_features - 1))
print("Adjusted R²:", adjusted_r2)

Adjusted R²: 0.7769085898923681


In [ ]:
# Interaction Feature

In [ ]:
# Features (X) and target (y)
X = insurance_data.drop(columns=["charges"])
y = insurance_data["charges"]

# One-Hot Encode 'region' (avoid dummy variable trap)
X = pd.get_dummies(
    X,
    columns=["region"],
    drop_first=True,
    dtype=int
)

# Binary encoding for categorical features
X["sex"] = X["sex"].map({"female": 1, "male": 0})
X["smoker"] = X["smoker"].map({"yes": 1, "no": 0})

# Feature Engineering: interaction terms
# Capture how smoking impacts charges across age and BMI
X["age_smoker"] = X["age"] * X["smoker"]
X["bmi_smoker"] = X["bmi"] * X["smoker"]

In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.22, random_state=42
)

# Train Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
  
r2 = r2_score(y_test, y_pred)
print("R² Score:", r2)

R² Score: 0.8594009865683826


In [23]:
# Adjusted R²: penalizes unnecessary features
n_samples = X_test.shape[0]
n_features = X_test.shape[1]

adjusted_r2 = 1 - ((1 - r2) * (n_samples - 1) / (n_samples - n_features - 1))
print("Adjusted R²:", adjusted_r2)

Adjusted R²: 0.8544503170813538


In [24]:
# Check for underfitting or overfitting
# Similar train & test R² (both low) -> Underfitting
# Much higher train R² than test R² -> Overfitting

y_train_pred = model.predict(X_train)
r2_train = r2_score(y_train, y_train_pred)

print("Training R²:", r2_train)
print("Test R²:", r2)

Training R²: 0.8350405366933565
Test R²: 0.8594009865683826
